# Идентификация пород собак
Этот ноутбук содержит базовое решение для соревнования Dog Breed Identification.

In [1]:
import os
import shutil
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import kagglehub
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
import timm
from torch.cuda.amp import autocast, GradScaler

## 1. Загрузка и настройка датасета

In [3]:
from kagglehub.auth import get_username

username = get_username()

if username is None:
    print("Вы не авторизованы в Kaggle. Запуск процесса авторизации...")
    kagglehub.login()
    raise RuntimeError("Пожалуйста, введите ваш API-токен Kaggle в интерактивном окне выше и запустите эту ячейку снова.")

print(f"Вы авторизованы как пользователь: {username}")

Вы авторизованы как пользователь: feytox


In [4]:
path = kagglehub.competition_download('dog-breed-identification')

DATA_DIR = Path(path)
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
LABELS_CSV = DATA_DIR / "labels.csv"
SAMPLE_SUBMISSION_CSV = DATA_DIR / "sample_submission.csv"

## 2. Разведочный анализ данных (EDA)
Анализ распределения классов и проверка размеров изображений.

In [5]:
df = pd.read_csv(LABELS_CSV)
print(f"Размер датасета: {df.shape}")
print(df.head())

counts = df["breed"].value_counts()
print(f"Минимум изображений на породу: {counts.min()}")
print(f"Максимум изображений на породу: {counts.max()}")

plt.figure(figsize=(12, 6))
counts.head(20).plot(kind="bar")
plt.title("Топ-20 самых популярных пород")
plt.ylabel("Количество изображений")
plt.tight_layout()
plt.show()

img_id = df.iloc[0]["id"]
img_path = TRAIN_DIR / f"{img_id}.jpg"
img = Image.open(img_path)
print(f"Пример изображения: {img_id}.jpg, размер: {img.size}, каналы: {img.mode}")

Размер датасета: (10222, 2)
                                 id             breed
0  000bec180eb18c7604dcecc8fe0dba07       boston_bull
1  001513dfcb2ffafc82cccf4d8bbaba97             dingo
2  001cdf01b096e06d78e9e5112d419397          pekinese
3  00214f311d5d2247d5dfe4fe24b2303d          bluetick
4  0021f9ceb3235effd7fcde7f7538ed62  golden_retriever
Минимум изображений на породу: 66
Максимум изображений на породу: 126
Пример изображения: 000bec180eb18c7604dcecc8fe0dba07.jpg, размер: (500, 375), каналы: RGB


C:\Users\Fey\AppData\Local\Temp\ipykernel_3628\1436005309.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Разделение выборки
Разделение данных на обучающую и валидационную выборки с использованием стратификации для сохранения соотношения классов.

In [6]:
breeds = sorted(df["breed"].unique())
breed_to_idx = {b: i for i, b in enumerate(breeds)}
idx_to_breed = {i: b for i, b in enumerate(breeds)}
df["label"] = df["breed"].map(breed_to_idx)

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

print(f"Размер обучающей выборки: {train_df.shape[0]}")
print(f"Размер валидационной выборки: {val_df.shape[0]}")

Размер обучающей выборки: 8177
Размер валидационной выборки: 2045


## 4. Класс датасета и загрузчики данных
Определение класса `DogDataset` для загрузки изображений и применение аугментаций для обучения.

In [7]:
class DogDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.ids = df["id"].values
        self.labels = df["label"].values if "label" in df.columns else None

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_name = f"{self.ids[idx]}.jpg"
        img_path = self.img_dir / img_name
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        if self.labels is not None:
            return image, self.labels[idx]
        return image, self.ids[idx]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = DogDataset(train_df, TRAIN_DIR, transform=train_transforms)
val_dataset = DogDataset(val_df, TRAIN_DIR, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

## 5. Инициализация предобученной модели ResNet-34
Загрузка ResNet-34 с весами ImageNet и изменение финального полносвязного слоя для классификации 120 пород собак.

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = timm.create_model("tf_efficientnetv2_s", pretrained=True, num_classes=120)
model = model.to(device)

print(f"Модель загружена и перемещена на: {device}")

Модель загружена и перемещена на: cuda


## 6. Функция потерь и оптимизатор

In [9]:
criterion = nn.CrossEntropyLoss()

backbone_params = []
classifier_params = []
for name, param in model.named_parameters():
    if "classifier" in name:
        classifier_params.append(param)
    else:
        backbone_params.append(param)

optimizer = optim.AdamW([
    {"params": backbone_params, "lr": 1e-5},
    {"params": classifier_params, "lr": 1e-4}
], weight_decay=1e-2)

scaler = GradScaler()

epochs = 10
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[1e-4, 1e-3],
    steps_per_epoch=len(train_loader),
    epochs=epochs,
    pct_start=0.2
)

C:\Users\Fey\AppData\Local\Temp\ipykernel_3628\3100351026.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


## 7. Цикл обучения и валидации

In [10]:
best_loss = float("inf")

for epoch in range(epochs):
    model.train()
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data).item()
        total += labels.size(0)
        
    epoch_train_loss = running_loss / total
    epoch_train_acc = correct / total
    
    model.eval()
    
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
                
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels.data).item()
            val_total += labels.size(0)
            
    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total
    
    print(f"Эпоха {epoch+1}/{epochs}")
    print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
    print(f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")
    
    if epoch_val_loss < best_loss:
        best_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model.pth")
        print("Сохранена лучшая модель.")

C:\Users\Fey\AppData\Local\Temp\ipykernel_3628\2830364801.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
C:\Users\Fey\AppData\Local\Temp\ipykernel_3628\2830364801.py:23: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
C:\Users\Fey\AppData\Local\Temp\ipykernel_3628\2830364801.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Эпоха 1/10
Train Loss: 4.4221 | Train Acc: 0.1156
Val Loss: 2.0369 | Val Acc: 0.4934
Сохранена лучшая модель.
Эпоха 2/10
Train Loss: 1.0250 | Train Acc: 0.7118
Val Loss: 0.7417 | Val Acc: 0.7716
Сохранена лучшая модель.
Эпоха 3/10
Train Loss: 0.3754 | Train Acc: 0.8789
Val Loss: 0.6902 | Val Acc: 0.7936
Сохранена лучшая модель.
Эпоха 4/10
Train Loss: 0.1627 | Train Acc: 0.9506
Val Loss: 0.6760 | Val Acc: 0.7971
Сохранена лучшая модель.
Эпоха 5/10
Train Loss: 0.0825 | Train Acc: 0.9773
Val Loss: 0.6691 | Val Acc: 0.8073
Сохранена лучшая модель.
Эпоха 6/10
Train Loss: 0.0512 | Train Acc: 0.9874
Val Loss: 0.6374 | Val Acc: 0.8196
Сохранена лучшая модель.
Эпоха 7/10
Train Loss: 0.0361 | Train Acc: 0.9941
Val Loss: 0.6315 | Val Acc: 0.8274
Сохранена лучшая модель.
Эпоха 8/10
Train Loss: 0.0262 | Train Acc: 0.9951
Val Loss: 0.6197 | Val Acc: 0.8308
Сохранена лучшая модель.
Эпоха 9/10
Train Loss: 0.0225 | Train Acc: 0.9952
Val Loss: 0.6100 | Val Acc: 0.8323
Сохранена лучшая модель.
Эпоха 10/1

## 8. Инференс на тестовой выборке и генерация сабмишна

In [11]:
test_df = pd.read_csv(SAMPLE_SUBMISSION_CSV)
test_dataset = DogDataset(test_df, TEST_DIR, transform=val_transforms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

model.load_state_dict(torch.load("best_model.pth"))
model.eval()

preds_list = []

with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)
        
        with autocast():
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            
        preds_list.append(probs.cpu().numpy())

preds_array = np.vstack(preds_list)

submission = pd.DataFrame(preds_array, columns=breeds)
submission.insert(0, "id", test_df["id"].values)
submission.to_csv("submission.csv", index=False)

print("Файл submission.csv успешно сохранен.")

C:\Users\Fey\AppData\Local\Temp\ipykernel_3628\2952503644.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Файл submission.csv успешно сохранен.


## 9. Проверка файла сабмишна

In [12]:
sub = pd.read_csv("submission.csv")
print(f"Формат сабмишна: {sub.shape}")
print(sub.head())
print(f"Есть ли пропущенные значения: {sub.isnull().any().any()}")

row_sums = sub.iloc[:, 1:].sum(axis=1)
print(f"Минимальная сумма строки: {row_sums.min():.4f}")
print(f"Максимальная сумма строки: {row_sums.max():.4f}")

Формат сабмишна: (10357, 121)
                                 id  affenpinscher  afghan_hound  \
0  000621fb3cbb32d8935728e48679680e   2.691224e-07  9.159165e-06   
1  00102ee9d8eb90812350685311fe5890   2.138262e-10  7.793272e-08   
2  0012a730dfa437f5f3613fb75efcd4ce   2.114880e-09  3.465161e-05   
3  001510bc8570bbeee98c8d80c8a95ec1   9.686382e-05  3.181302e-06   
4  001a5f3114548acdefa3d4da05474c2e   5.960749e-03  1.007241e-04   

   african_hunting_dog      airedale  american_staffordshire_terrier  \
0         1.697334e-07  6.581278e-09                    7.983996e-10   
1         2.017176e-08  1.004293e-09                    4.163920e-07   
2         2.345885e-08  3.156734e-08                    4.398531e-10   
3         7.705287e-07  7.871446e-08                    4.109491e-05   
4         2.066416e-05  2.540976e-07                    1.288088e-05   

    appenzeller  australian_terrier       basenji        basset  ...  \
0  1.187069e-08        1.497892e-07  7.271682e-08  1.062